In [ ]:
# Dependencies are installed in the active notebook kernel.
# Keep this cell as a no-op so the notebook stays runnable locally.


In [ ]:
import geemap
import ee
import pandas as pd
import geopandas as gpd

# Authenticate once in this machine/session before running the notebook end-to-end.
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com',
)

# Set the path to the JSON file containing the geometry.
path_json = "../data/bacias_meso_SF.geojson"
gdf = gpd.read_file(path_json)
gdf_menor = gdf[gdf['nm_mesoRH'] == 'Baixo São Francisco']
geom_ee = geemap.geopandas_to_ee(gdf_menor)
area = geom_ee.geometry()

# 3. Carregar a coleção MODIS (Reflectância de Superfície)
modis = ee.ImageCollection('MODIS/061/MOD09A1') \
    .filterBounds(area) \
    .filterDate('2000-02-18', '2025-12-31')

In [4]:
# 4. Função para calcular MSAVI e Albedo
def add_indices(image):
    # Aplicar fator de escala do MODIS para obter a reflectância real [7]
    img_scaled = image.multiply(0.0001)

    # Cálculo do MSAVI usando .expression() [4]
    msavi = img_scaled.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': img_scaled.select('sur_refl_b02'), # Banda NIR no MODIS
            'RED': img_scaled.select('sur_refl_b01')  # Banda RED no MODIS
        }).rename('MSAVI')

    # Cálculo do Albedo (Exemplo usando fórmula empírica comum para MODIS)
    # Verifique os coeficientes exatos da metodologia que você está seguindo
    albedo = img_scaled.expression(
        '0.160 * B1 + 0.291 * B2 + 0.243 * B3 + 0.116 * B4 + 0.112 * B5 + 0.081 * B7 - 0.0015', {
            'B1': img_scaled.select('sur_refl_b01'),
            'B2': img_scaled.select('sur_refl_b02'),
            'B3': img_scaled.select('sur_refl_b03'),
            'B4': img_scaled.select('sur_refl_b04'),
            'B5': img_scaled.select('sur_refl_b05'),
            'B7': img_scaled.select('sur_refl_b07')
        }).rename('Albedo')

    # Adiciona as novas bandas à imagem original e mantém as propriedades de data [4, 8]
    return image.addBands([msavi, albedo]).copyProperties(image, ['system:time_start'])

# 1. Função para extrair a média e a data de cada imagem
def extrair_serie(image):
    # Calcula a média do MSAVI e Albedo dentro do seu ROI
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=500, # Resolução nativa do MODIS em metros
        maxPixels=1e9
    )

    # Retorna os dados como atributos de uma Feature (sem a geometria pesada)
    return ee.Feature(None, {
        'Data': image.date().format('YYYY-MM-dd'),
        'MSAVI': stats.get('MSAVI'),
        'Albedo': stats.get('Albedo')
    })

In [5]:
# 5. Mapear a função sobre toda a coleção de imagens
modis_com_indices = modis.map(add_indices)

# Testar a performance no XEE

In [7]:
# 1. Inicializar o XEE
import xarray as xr
from xee import helpers
import geopandas as gpd

aoi = gdf_menor.geometry.union_all()

# Definir parâmetros para fit da geometria
GRID_CRS = 'EPSG:6933'
AOI_CRS = 'EPSG:4326'
GRID_SCALE = (1000, -1000)

grid_params = helpers.fit_geometry(
    geometry=aoi,
    geometry_crs=AOI_CRS,
    grid_crs=GRID_CRS,
    grid_scale=GRID_SCALE,
)

# Filtrar a coleção apenas com as bandas de interesse (MSAVI e Albedo)
modis_com_indices = modis_com_indices.select(['MSAVI', 'Albedo'])

# Abre a coleção do GEE como um cubo de dados multidimensional
ds = xr.open_dataset(
    modis_com_indices,
    engine='ee',
    **grid_params,
    chunks='auto',
)

ds = ds.mean(dim='time') * 1


## ABORDAGEM 1: DDI (Desertification Divided Index) - Regressão Linear
- Utilizando o coeficiente empírico (K = 1.803) validado na literatura [11]
- Valores MAIORES indicam áreas severamente preservadas (ou não-desertificadas, dependendo do sinal do índice).
- A literatura em [11] frequentemente inverte o eixo para mapear Intensidade I.
- No modelo padrão DDI (K * MSAVI - Albedo), o decréscimo representa degradação [10].

## ABORDAGEM 2: SASDI (Point-to-Point Distance Model)
- Calcula a distância euclidiana do pixel em relação ao estado ideal (MSAVI=1, Albedo=0) [12, 13]
- Quanto MAIOR a distância, MAIOR o grau de desertificação.
- O xarray aplica a operação vetorial elemento a elemento perfeitamente no Dask Graph.
- ---------------------------------------------------------

In [8]:
import numpy as np

K_coef = 1.803
ds['DDI'] = (K_coef * ds['MSAVI']) - ds['Albedo']


ds['SASDI'] = np.sqrt((ds['MSAVI'] - 1)**2 + (ds['Albedo'])**2)

In [9]:
ds

<xarray.Dataset> Size: 782kB
Dimensions:  (y: 295, x: 165)
Coordinates:
  * y        (y) float64 2kB -1.056e+06 -1.058e+06 ... -1.35e+06 -1.35e+06
  * x        (x) float64 1kB -3.666e+06 -3.666e+06 ... -3.504e+06 -3.502e+06
Data variables:
    MSAVI    (y, x) float32 195kB dask.array<chunksize=(295, 165), meta=np.ndarray>
    Albedo   (y, x) float32 195kB dask.array<chunksize=(295, 165), meta=np.ndarray>
    DDI      (y, x) float32 195kB dask.array<chunksize=(295, 165), meta=np.ndarray>
    SASDI    (y, x) float32 195kB dask.array<chunksize=(295, 165), meta=np.ndarray>

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, ax = plt.subplots(figsize=(10, 4), subplot_kw={'projection': ccrs.PlateCarree()})

# 1. Plotar a geometria da bacia
ax.add_geometries(gdf_menor.geometry, crs=ccrs.PlateCarree(), facecolor='none', edgecolor='black', linewidth=1)

# 2. Definir o zoom na bacia
bounds = gdf_menor.total_bounds # [minx, miny, maxx, maxy]
ax.set_extent([bounds[0], bounds[2], bounds[1], bounds[3]], crs=ccrs.PlateCarree())

# 3. Plotar o DDI com imshow rápido
# ATENÇÃO: Se o seu DataArray tiver dimensão 'time', selecione um tempo único com .isel(time=0)
ddi_plot = ds['DDI'].plot.imshow(
    ax=ax,
    transform=ccrs.EqualEarth(), # Ou ccrs.PlateCarree(), veja os detalhes abaixo!
    cmap='viridis',
    alpha=0.6,
    add_colorbar=True,
    robust=True # Ajusta o contraste ignorando outliers
)

### Visualização da Série Temporal dos Índices de Desertificação

Vamos plotar as séries temporais do DDI e SASDI para observar as tendências ao longo do tempo.